# Importação de bibliotecas

In [1]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E PACOTES
# ==============================================================================
# 1. Bibliotecas de terceiros (manipulação de dados)
import pandas as pd
import numpy as np

# 2. Bibliotecas de Processamento de Linguagem Natural (NLTK)
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 3. Ferramentas do scikit-learn (Vetorização e Similaridade)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ==============================================================================
# 2. CONFIGURAÇÃO DE CAMINHOS (PATHS) E PARÂMETROS GERAIS
# ==============================================================================
# 1. Definição de caminhos base e entradas
BASE_PATH = '../data/processed'
INPUT_FILE = f'{BASE_PATH}/processed_publications.csv'

# 2. Parametrização do modelo NLP
NGRAM_RANGE = (1,1)  # (1,1)=Unigrama, (2,2)=Bigrama, (1,2)=Misto
STOPWORD_LANGUAGES = ['portuguese', 'english', 'spanish']

# 3. Dicionário e configuração de n-gramas
NGRAM_CONFIGS = {
    (1,1) : 'unigram', # Unigrama
    (2,2) : 'bigram',  # Bigrama
    (1,2) : 'mixed'    # Misto
}

CURRENT_CONFIG = NGRAM_CONFIGS[NGRAM_RANGE]
print(f'Configuração atual: {CURRENT_CONFIG}.')

# 4. Definição de caminhos de saída (Outputs)
OUTPUT_SIM_PARQUET = f'{BASE_PATH}/similarity_matrices/similarity_{CURRENT_CONFIG}.parquet'
OUTPUT_WORDS_PARQUET = f'{BASE_PATH}/word_matrices/words_{CURRENT_CONFIG}.parquet'
# OUTPUT_SIM_CSV = f'{BASE_PATH}/similarity_matrices/similarity_{CURRENT_CONFIG}.csv'
# OUTPUT_WORDS_CSV = f'{BASE_PATH}/word_matrices/words_{CURRENT_CONFIG}.csv'

Configuração atual: unigram.


# Definição das stopwords

In [3]:
# ==============================================================================
# 3. CARREGAMENTO E DEFINIÇÃO DO CONJUNTO DE STOPWORDS
# ==============================================================================
print(f">>> Carregando stopwords para: {STOPWORD_LANGUAGES}...")

stop_words = set()

# 1. Varredura e consolidação de stopwords por idioma
for language in STOPWORD_LANGUAGES:
    try:
        # Acrescenta as stopwords de cada língua ao conjunto
        stop_words.update(stopwords.words(language))
    except OSError:
        print(f"Aviso: O pacote de stopwords para '{language}' não foi encontrado.")

print(f"✅ Total de stopwords carregadas: {len(stop_words)}")

>>> Carregando stopwords para: ['portuguese', 'english', 'spanish']...
✅ Total de stopwords carregadas: 679


# Lematização e processamento dos títulos

In [4]:
# ==============================================================================
# 4. FUNÇÕES DE PRÉ-PROCESSAMENTO (NLP PIPELINE)
# ==============================================================================
# 1. Instancia o lematizador global
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Pipeline de Limpeza de Texto (NLP):
    # O objetivo é reduzir o ruído e manter apenas o núcleo semântico.
    if not isinstance(text, str):
        return ""

    # 1. Lowercase: padroniza para minúsculas.
    # 2. Tokenização: quebra o texto em palavras.
    # 3. Filtragem: remove números/pontuação (isalpha) e stopwords (palavras comuns sem valor semântico).
    # 4. Lemmatização: reduz palavras à sua raiz (ex: "estudos" -> "estudo") para agrupar termos similares.
    text = text.lower()
    tokens = [lemmatizer.lemmatize(word) for word in word_tokenize(text) if word.isalpha() and word not in stop_words]
    return " ".join(tokens)

# Carregamento dos dados

In [5]:
# ==============================================================================
# 5. CARREGAMENTO DA BASE DE DADOS BRUTA (LOAD)
# ==============================================================================
# 1. Carregamento dos dados curados manualmente
print(">>> Carregando base de dados bruta das publicações...")
df_publications = pd.read_csv(INPUT_FILE, sep=';')

print(f"✅ Total de registos carregados: {len(df_publications)}\n")

>>> Carregando base de dados bruta das publicações...
✅ Total de registos carregados: 999



In [6]:
# ==============================================================================
# 6. HIGIENE DO ARQUIVO E APLICAÇÃO DO NLP PIPELINE
# ==============================================================================
print(">>> Aplicando pipeline de NLP (tokenização e lematização) aos títulos...")

# 1. Remove linhas inteiramente vazias geradas pela exportação do Excel.
# Como a base foi curada manualmente, garantimos que todo registro válido possui um 'Pesquisador' preenchido.
df_publications = df_publications.dropna(subset=['Pesquisador'])

# 2. Garante tipagem correta (string) para evitar erros de leitura
df_publications['Título'] = df_publications['Título'].astype(str)

# 3. Aplica o pipeline do pré-processamento aos títulos
df_publications['Título_Limpo'] = df_publications['Título'].apply(preprocess_text)

print("✅ Textos processados e higienizados com sucesso!\n")

>>> Aplicando pipeline de NLP (tokenização e lematização) aos títulos...
✅ Textos processados e higienizados com sucesso!



# Vetorização por contagem (Count)

### Definição de n-gramas

Neste projeto, realizamos experimentos variando o intervalo de n-gramas (unigramas `(1,1)`, bigramas `(2,2)` e mistos `(1,2)`) para avaliar qual configuração capturava melhor as áreas de pesquisa sem introduzir ruído excessivo.

* **Configuração atual:** O notebook está parametrizado para gerar **unigramas** (palavras únicas).
* **Justificativa:** Esta configuração foi selecionada para esta execução específica pois, nos testes posteriores, demonstrou melhor desempenho estatístico.

In [7]:
# ==============================================================================
# 7. VETORIZAÇÃO POR CONTAGEM (COUNT VECTORIZER)
# ==============================================================================
# Utilizamos o CountVectorizer para converter títulos em vetores de frequência.
# Estratégia:
# - CountVectorizer vs TF-IDF: optamos pela contagem bruta pois, em títulos acadêmicos,
#   a repetição de termos técnicos é um sinal forte de especialização, que poderia
#   ser penalizado pelo IDF.
# - N-grams: Definido por NGRAM_RANGE (unigrama, bigrama ou misto)

print(f">>> Vetorizando textos com CountVectorizer (Configuração: {CURRENT_CONFIG})...")

# 1. Instancia o vetorizador com a configuração de N-gramas
vectorizer = CountVectorizer(ngram_range = NGRAM_RANGE)

# 2. Aprendizado do Vocabulário e Transformação:
#    - .fit(): aprende o vocabulário inteiro do corpus (todas as palavras únicas).
#    - .transform(): gera uma matriz esparsa, onde:
#       * Linhas = Pesquisadores.
#       * Colunas = Palavras do vocabulário.
#       * Valores = Frequência da palavra naquele pesquisador.
count_matrix = vectorizer.fit_transform(df_publications['Título_Limpo'])

print(f"✅ Matriz vetorizada com sucesso! Dimensões: {count_matrix.shape}\n")

>>> Vetorizando textos com CountVectorizer (Configuração: unigram)...
✅ Matriz vetorizada com sucesso! Dimensões: (89, 2547)



# Aplicação da similaridade de cosseno à matriz

## Objetivo:
Calcular a **Similaridade de Cosseno** entre todos os vetores de pesquisadores.

Fórmula: $$ \text{Similaridade} = \frac{\mathbf{A} \cdot \mathbf{B}}{\| \mathbf{A} \| \| \mathbf{B} \|} $$

## Resultado:
Uma matriz quadrada (NxN) onde:

* **1.0:** Vetores alinhados (Vocabulário idêntico).
* **0.0:** Vetores ortogonais (Nenhuma palavra em comum).

In [8]:
# ==============================================================================
# 8. CÁLCULO DA MATRIZ DE SIMILARIDADE DE COSSENO
# ==============================================================================
print(">>> Calculando a matriz de similaridade de cosseno entre investigadores...")

# 1. Calcula a matriz de distância NxN.
# A entrada é a matriz esparsa 'count_matrix' para otimizar o cálculo matricial.
similarity_matrix = cosine_similarity(count_matrix)

print("✅ Similaridade calculada com sucesso!\n")

>>> Calculando a matriz de similaridade de cosseno entre investigadores...
✅ Similaridade calculada com sucesso!



# Criação do DataFrame de similaridades

In [9]:
# ==============================================================================
# 9. ESTRUTURAÇÃO DO DATAFRAME DE SIMILARIDADES (MATRIZ DE ADJACÊNCIA)
# ==============================================================================
# 1. Extrai os nomes dos autores da relação de publicações
author_names = df_publications['Pesquisador'].tolist()

# 2. Cria o DataFrame com as similaridades indexadas pelos autores
similarity_matrix = pd.DataFrame(similarity_matrix, index=author_names, columns=author_names)

In [10]:
# ==============================================================================
# 10. VISUALIZAÇÃO DOS DADOS DE SIMILARIDADE
# ==============================================================================
print(f"Dimensões da matriz: {similarity_matrix.shape}")
similarity_matrix

Dimensões da matriz: (89, 89)


,Adriano Jose Ferruzzi,Alessandra da Silva Carneiro,Alessandro Emilio Teruzzi,Alex Sandro Rodrigues Ancioto,Alexandre José Romagnoli,Alexandre Machado Rosa,Ana Carolina Vila Ramos dos Santos,Ana Claudia Folha da Cruz,Ana Márcia Lima Costa,Ana Paula Rodrigues Magalhães de Barros,...,Silene Jucelino de Lima,Tatiana Aparecida Picosque,Teresa Helena Buscato Martins,Thiago Pedro Donadon Homem,Thiago Silva Broze,Valdir Donizete dos Santos Junior,Vanessa Regina Ferreira da Silva,Whisner Fraga Mamede,Wilian Ramalho Feitosa,William Rosseti
Adriano Jose Ferruzzi,1.000000,0.0,0.000000,0.036120,0.047782,0.000000,0.026851,0.056773,0.000000,0.008676,...,0.000000,0.028387,0.064923,0.027418,0.033787,0.014517,0.000000,0.0,0.063665,0.022630
Alessandra da Silva Carneiro,0.000000,1.0,0.000000,0.000000,0.000000,0.052705,0.015610,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
Alessandro Emilio Teruzzi,0.000000,0.0,1.000000,0.048795,0.032275,0.000000,0.000000,0.000000,0.163299,0.117202,...,0.000000,0.000000,0.043853,0.000000,0.000000,0.039223,0.000000,0.0,0.009556,0.030571
Alex Sandro Rodrigues Ancioto,0.036120,0.0,0.048795,1.000000,0.031497,0.000000,0.000000,0.037424,0.039841,0.057189,...,0.000000,0.000000,0.064194,0.036147,0.000000,0.000000,0.000000,0.0,0.018652,0.014917
Alexandre José Romagnoli,0.047782,0.0,0.032275,0.031497,1.000000,0.052705,0.039024,0.049507,0.026352,0.007565,...,0.000000,0.024754,0.070767,0.000000,0.036828,0.012659,0.000000,0.0,0.101782,0.078934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Valdir Donizete dos Santos Junior,0.014517,0.0,0.039223,0.000000,0.012659,0.032026,0.000000,0.000000,0.000000,0.027582,...,0.000000,0.000000,0.017201,0.019371,0.000000,1.000000,0.000000,0.0,0.003748,0.035973
Vanessa Regina Ferreira da Silva,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.110096,0.173838,...,0.000000,0.000000,0.000000,0.000000,0.015386,0.000000,1.000000,0.0,0.019329,0.041222
Whisner Fraga Mamede,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.029463,0.000000,0.000000,1.0,0.000000,0.000000
Wilian Ramalho Feitosa,0.063665,0.0,0.009556,0.018652,0.101782,0.046816,0.048530,0.014659,0.015605,0.022400,...,0.258022,0.029318,0.037717,0.089672,0.161389,0.003748,0.019329,0.0,1.000000,0.300911


# Obtendo palavras em comum entre os pesquisadores

## Objetivo:
Identificar quais palavras geraram a similaridade e utilizar posteriormente como labels das arestas do grafo.

## Otimização de performance:
Em vez de reprocessar as strings originais (o que seria lento, com complexidade $O(N \times M)$), utilizamos a própria estrutura da matriz já vetorizada.

## Lógica Matricial:
* **dense_matrix > 0:** Cria máscaras booleanas indicando a presença dos termos.
* **& (AND):** Encontra a interseção matemática imediata entre dois vetores.
* **feature_names:** Mapeia os índices resultantes de volta para palavras legíveis.

In [11]:
# ==============================================================================
# 11. PREPARAÇÃO DE VARIÁVEIS PARA EXTRAÇÃO DE FEATURES
# ==============================================================================
# 1. Recuperação do vocabulário:
# Obtém a lista de palavras (ou n-grams) correspondente a cada coluna da matriz.
feature_names = vectorizer.get_feature_names_out()

# 2. Conversão de matriz esparsa para densa:
# O fit_transform gerou uma matriz esparsa (para economizar memória).
# Aqui, convertemos para array denso para permitir acesso rápido via índices no loop.
dense_matrix = count_matrix.toarray()

n_authors = len(author_names)

# 3. Cria uma matriz vazia para abrigar as futuras strings das palavras em comum
common_words_matrix = [['' for _ in range(n_authors)] for _ in range(n_authors)]

In [12]:
# ==============================================================================
# 12. GERAÇÃO DA MATRIZ DE PALAVRAS COMPARTILHADAS (INTERSEÇÃO MATRICIAL)
# ==============================================================================
print(">>> Gerando matriz de palavras em comum...")

# 1. Varredura par a par de autores
for i in range(n_authors):
    for j in range(n_authors):
        # 2. Ignora a diagonal principal (comparação de pesquisador consigo mesmo)
        if i == j:
            continue

        # 3. Encontra palavras em comum entre o par usando array index mapping
        # Usando [0] para extrair o array de índices da tupla retornada pelo numpy
        intersection_indices = np.where((dense_matrix[i] > 0) & (dense_matrix[j] > 0))[0]

        # 4. Traduz índices numéricos para palavras legíveis do vocabulário
        common_words = [feature_names[idx] for idx in intersection_indices]

        # 5. Formata como texto separado por vírgulas
        common_words_matrix[i][j] = ", ".join(common_words)

print('✅ Matriz gerada e preenchida com sucesso!\n')

>>> Gerando matriz de palavras em comum...
✅ Matriz gerada e preenchida com sucesso!



# Criação do DataFrame de palavras

In [13]:
# ==============================================================================
# 13. ESTRUTURAÇÃO DO DATAFRAME DE PALAVRAS COMPARTILHADAS
# ==============================================================================
# 1. Converte a matriz (lista de listas) em um DataFrame pandas estruturado
df_shared_terms = pd.DataFrame(common_words_matrix, index=author_names, columns=author_names)

# 2. Visualização das dimensões e da matriz
print(f"Dimensões da matriz: {df_shared_terms.shape}")
df_shared_terms

Dimensões da matriz: (89, 89)


,Adriano Jose Ferruzzi,Alessandra da Silva Carneiro,Alessandro Emilio Teruzzi,Alex Sandro Rodrigues Ancioto,Alexandre José Romagnoli,Alexandre Machado Rosa,Ana Carolina Vila Ramos dos Santos,Ana Claudia Folha da Cruz,Ana Márcia Lima Costa,Ana Paula Rodrigues Magalhães de Barros,...,Silene Jucelino de Lima,Tatiana Aparecida Picosque,Teresa Helena Buscato Martins,Thiago Pedro Donadon Homem,Thiago Silva Broze,Valdir Donizete dos Santos Junior,Vanessa Regina Ferreira da Silva,Whisner Fraga Mamede,Wilian Ramalho Feitosa,William Rosseti
Adriano Jose Ferruzzi,,,,análise,análise,,"acesso, redes, sociais",análise,,system,...,,acesso,análise,aplicação,"aplicação, dado",apresentação,,,"acesso, análise, clientes, dado, meio, redes, ...",análise
Alessandra da Silva Carneiro,,,,,,nova,mulher,,,,...,,,,,,,,,,
Alessandro Emilio Teruzzi,,,,ensino,ideias,,,,"ensino, médio","ensino, matemática",...,,,ensino,,,história,,,história,história
Alex Sandro Rodrigues Ancioto,análise,,ensino,,análise,,,análise,ensino,"ensino, teaching",...,,,"análise, ensino",simulator,,,,,"análise, evaluation, virtual",análise
Alexandre José Romagnoli,análise,,ideias,análise,,"educação, papel","educação, política",análise,educação,produção,...,,educação,"análise, produção",,"ferramenta, gestão, papel, política",disputa,,,"análise, brasil, educação, gestão, papel, sist...","análise, brasil, gestão"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Valdir Donizete dos Santos Junior,apresentação,,história,,disputa,nação,,,,experiência,...,,,experiência,"experiência, máscaras",,,,,história,"história, indústria, perspectivas"
Vanessa Regina Ferreira da Silva,,,,,,,,,"cursos, reflexões","docente, olhar, reflexões",...,,,,,trabalho,,,,paulo,paulo
Whisner Fraga Mamede,,,,,,,,,,,...,,,,,utilizando,,,,,
Wilian Ramalho Feitosa,"acesso, análise, clientes, dado, meio, redes, ...",,história,"análise, evaluation, virtual","análise, brasil, educação, gestão, papel, sist...","construção, educação, papel","acesso, cidade, cultura, educação, ifsp, polít...","análise, estratégia","educação, impactos","avaliação, campus, context, contexto, digital,...",...,"atendimento, caso, estudo, hospital, operacion...","acesso, educação, políticas","análise, campus, conhecimento, construção, ifs...","approach, based, case, comunicação, conhecimen...","brazil, case, caso, dado, desenvolvimento, est...",história,paulo,,,"acessibilidade, análise, aérea, bank, brasil, ..."


# Exportação dos resultados

In [14]:
# ==============================================================================
# 14. EXPORTAÇÃO DOS RESULTADOS MATRICIAIS (LOAD)
# ==============================================================================
# 1. Persistência da matriz de similaridades para os próximos notebooks
similarity_matrix.to_parquet(OUTPUT_SIM_PARQUET)
print(f'✅ Matriz de similaridades salva com sucesso como {OUTPUT_SIM_PARQUET}')

# 2. Persistência da matriz de palavras para uso posterior na rede
df_shared_terms.to_parquet(OUTPUT_WORDS_PARQUET)
print(f'✅ Matriz de palavras salva com sucesso como {OUTPUT_WORDS_PARQUET}')

✅ Matriz de similaridades salva com sucesso como ../data/processed/similarity_matrices/similarity_unigram.parquet
✅ Matriz de palavras salva com sucesso como ../data/processed/word_matrices/words_unigram.parquet


In [15]:
# ==============================================================================
# 15. ROTINAS DE AUDITORIA OPCIONAIS
# ==============================================================================
# 1. Exportação manual em formato Excel para verificações (descomentar se necessário)

# Arredondar para 4 casas decimais para evitar erro de visualização dos valores
# similarity_matrix_rounded = similarity_matrix.round(4)

# similarity_matrix_rounded.to_csv(OUTPUT_SIM_CSV, sep=';', decimal=',', encoding='utf-8-sig', index=True)
# print(f'Matriz de similaridades salva com sucesso como {OUTPUT_SIM_CSV}')

# df_shared_terms.to_csv(OUTPUT_WORDS_CSV, sep=';', decimal=',', encoding='utf-8-sig', index=True)
# print(f'Matriz de palavras salva com sucesso como {OUTPUT_WORDS_CSV}')